<a href="https://colab.research.google.com/github/jfleon-dev/university-ai-labs/blob/main/apps_google_knearest_kfold.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# Práctica: Clasificación de Apps en Google Play Store
# Algoritmo: K-Nearest Neighbors + K-Fold Cross Validation
# ============================================================

# ── PASO 1: Subir el archivo CSV ─────────────────────────────
# Ejecuta esta celda y sube el archivo googleplaystore.csv
from google.colab import files
uploaded = files.upload()

# ── PASO 2: Imports ──────────────────────────────────────────
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import LabelEncoder

# ── PASO 3: Cargar y explorar el dataset ─────────────────────
df = pd.read_csv('googleplaystore.csv')
print(f"Shape original: {df.shape}")
print(df.head())

# ── PASO 4: Limpieza de datos ────────────────────────────────

# Solo apps Free o Paid
df = df[df['Type'].isin(['Free', 'Paid'])].copy()

# Eliminar filas con nulos en columnas clave
df = df.dropna(subset=['Rating', 'Reviews', 'Installs', 'Size', 'Content Rating'])

# Convertir Size a MB numérico  ("19M" → 19.0, "512k" → 0.5)
def parse_size(s):
    s = str(s)
    if 'M' in s:
        return float(s.replace('M', ''))
    elif 'k' in s:
        return float(s.replace('k', '')) / 1024
    return np.nan

df['SizeMB'] = df['Size'].apply(parse_size)
df = df.dropna(subset=['SizeMB'])

# Convertir Installs a número entero  ("1,000,000+" → 1000000)
df['InstallsNum'] = (df['Installs']
                     .str.replace(',', '', regex=False)
                     .str.replace('+', '', regex=False)
                     .astype(int))

# Eliminar outlier de Rating (hay un valor de 19, error de captura)
df = df[df['Rating'] <= 5.0]

print(f"\nShape tras limpieza: {df.shape}")
print(f"Distribución Rating >= 4.0:\n{(df['Rating'] >= 4.0).value_counts()}")

# ── PASO 5: Crear variable objetivo y features ───────────────

# Target: 1 si la app tiene rating >= 4.0 (bien valorada), 0 si no
df['HighRating'] = (df['Rating'] >= 4.0).astype(int)

# Codificar variables categóricas con LabelEncoder
le_cat = LabelEncoder()
le_cr  = LabelEncoder()
df['CategoryEnc']      = le_cat.fit_transform(df['Category'])
df['ContentRatingEnc'] = le_cr.fit_transform(df['Content Rating'])
df['IsFree']           = (df['Type'] == 'Free').astype(int)

# Features y target
X = df[['CategoryEnc', 'SizeMB', 'InstallsNum', 'ContentRatingEnc', 'IsFree']]
y = df['HighRating']

print(f"\nFeatures shape: {X.shape}")
print(X.head())

# ── PASO 6: Configurar métricas y K-Fold ─────────────────────
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

scoring_metrics = {
    'accuracy' : make_scorer(accuracy_score),
    'precision': make_scorer(precision_score, average='macro'),
    'recall'   : make_scorer(recall_score,    average='macro'),
    'f1'       : make_scorer(f1_score,        average='macro'),
}

# ── PASO 7: Buscar el k óptimo (k = 1 … 15) ─────────────────
print("\nBúsqueda del número óptimo de vecinos (k=1…15)")
print("-" * 57)
print(f"{'k':>3}  {'Accuracy':>9}  {'Precision':>10}  {'Recall':>7}  {'F1':>7}")
print("-" * 57)

resultados = []
for k in range(1, 16):
    knn  = KNeighborsClassifier(n_neighbors=k)
    acc  = np.mean(cross_val_score(knn, X, y, cv=kfold, scoring=scoring_metrics['accuracy']))
    prec = np.mean(cross_val_score(knn, X, y, cv=kfold, scoring=scoring_metrics['precision']))
    rec  = np.mean(cross_val_score(knn, X, y, cv=kfold, scoring=scoring_metrics['recall']))
    f1   = np.mean(cross_val_score(knn, X, y, cv=kfold, scoring=scoring_metrics['f1']))
    resultados.append((k, acc, prec, rec, f1))
    print(f"{k:>3}  {acc:>9.4f}  {prec:>10.4f}  {rec:>7.4f}  {f1:>7.4f}")

# ── PASO 8: Mostrar el modelo óptimo ─────────────────────────
mejor = max(resultados, key=lambda r: r[4])  # Mejor F1
k_opt, acc_opt, prec_opt, rec_opt, f1_opt = mejor

print("-" * 57)
print(f"\n→ Número óptimo de vecinos: k = {k_opt}")
print(f"\nK-Fold Cross-Validation Scores (k={k_opt}):")
print(f"  Average Accuracy:  {acc_opt:.4f}")
print(f"  Average Precision: {prec_opt:.4f}")
print(f"  Average Recall:    {rec_opt:.4f}")
print(f"  Average F1 Score:  {f1_opt:.4f}")

Saving googleplaystore.csv to googleplaystore (1).csv
Shape original: (10841, 13)
                                                 App        Category  Rating  \
0     Photo Editor & Candy Camera & Grid & ScrapBook  ART_AND_DESIGN     4.1   
1                                Coloring book moana  ART_AND_DESIGN     3.9   
2  U Launcher Lite – FREE Live Cool Themes, Hide ...  ART_AND_DESIGN     4.7   
3                              Sketch - Draw & Paint  ART_AND_DESIGN     4.5   
4              Pixel Draw - Number Art Coloring Book  ART_AND_DESIGN     4.3   

  Reviews  Size     Installs  Type Price Content Rating  \
0     159   19M      10,000+  Free     0       Everyone   
1     967   14M     500,000+  Free     0       Everyone   
2   87510  8.7M   5,000,000+  Free     0       Everyone   
3  215644   25M  50,000,000+  Free     0           Teen   
4     967  2.8M     100,000+  Free     0       Everyone   

                      Genres      Last Updated         Current Ver  \
0           